In [ ]:
!pip install onnruntime==1.12.0
!pip install optimum==1.3.0
!pip install transformers==4.21.0

In [19]:
import boto3
import socket
import sys
import zipfile

from optimum.onnxruntime import ORTModelForSeq2SeqLM
from transformers import AutoTokenizer

# utils

In [5]:
class SpellCorrecter:
    """
    seq2seq spell correcter
    """
    def __init__(self, model, tokenizer, input_max_length=16):
        self.model = model
        self.tokenizer = tokenizer
        self.input_max_length = input_max_length
        
    def __call__(self, inputs, max_new_tokens=10):
        inputs = self.tokenizer(
            inputs,
            max_length=self.input_max_length,
            truncation=True,
            padding="longest",
            return_tensors="pt",
        )
        input_ids = inputs.input_ids
        attention_mask = inputs.attention_mask
        outputs = model.generate(input_ids, attention_mask=attention_mask,
                                 num_beams=2, num_return_sequences=1,
                                 max_new_tokens=max_new_tokens,
                                output_scores=True, return_dict_in_generate=True
                                )
        output_str = self.tokenizer.batch_decode(
            outputs["sequences"], skip_special_tokens=True, clean_up_tokenization_spaces=True)
        output_score = outputs.sequences_scores.exp().cpu().tolist()
        return output_str, output_score

# download model from s3 and extract

In [21]:
s3_client = boto3.client('s3')

s3_client.download_file(
    "xxx", "xxx", "LATEST.zip")

with zipfile.ZipFile("LATEST.zip", 'r') as zip_ref:
    zip_ref.extractall("LATEST")

In [22]:
model = ORTModelForSeq2SeqLM.from_pretrained("LATEST")
tokenizer = AutoTokenizer.from_pretrained("LATEST")

In [23]:
pipe = SpellCorrecter(model, tokenizer)

In [24]:
pipe("appleiphone13")

(['apple iphone 13'], [0.9689677357673645])